# 02 · Raster analytics with GeoBrix (lightweight tier, serverless)

**What this shows:** multidimensional NetCDF (variable × scenario × decade × month) read into
Delta as raster tiles, **normalised** (Kelvin → °C, kg m⁻² s⁻¹ → mm/day), **reduced** (decadal means), **differenced**
against a baseline, **clipped** to Victoria, exported as **Cloud-Optimised GeoTIFFs for ArcGIS**, and finally
**gridded to H3** so the rasters can be joined to anything in SQL.

```
NetCDF (14 files) ──netcdf_gbx──▶ 1,680 monthly tiles ──rst_mapalgebra──▶ °C / mm·day⁻¹
       │                                     │
       │                     rst_combineavg_agg ▶ decadal means ──rst_mapalgebra(A-B)──▶ change rasters ──▶ COG (ArcGIS)
       │                                                                                      └─rst_clip──▶ Victoria
       └───────────────────────────── gbx_rst_h3_rastertogridavg ──▶ H3 cell × month table (SQL-ready)
```
GeoBrix is a **Databricks Labs** project (no SLA). The lightweight tier is pure Python (rasterio), with no JAR and no init script, so it runs on serverless.

In [ ]:
%pip install -q "geobrix[light_env6] @ https://github.com/databrickslabs/geobrix/releases/download/v0.5.2/geobrix-0.5.2-py3-none-any.whl"

In [ ]:
%restart_python

In [ ]:
%run ./00_config

In [ ]:
from pyspark.sql import functions as F
from databricks.labs.gbx.ds.register import register
from databricks.labs.gbx.pyrx import functions as rx

register(spark)      # *_gbx readers / writers
rx.register(spark)   # gbx_rst_* SQL functions

## 1 · Read every NetCDF file: one tile per (variable, scenario, decade, month)
`fanout=time` expands the 120 monthly slices in each decade file into separate rows: **14 files → 1,680 tiles**.

In [ ]:
AUS_WKT = "POLYGON((112 -44.5, 154.5 -44.5, 154.5 -9.5, 112 -9.5, 112 -44.5))"   # crop the AUS-20i domain to Australia

raw = spark.read.format("netcdf_gbx").option("fanout", "time").load(f"{VOL}/barpa/")

pattern = r"/(\w+)_AUS-20i_ACCESS-CM2_(\w+)_r4i1p1f1_BOM_BARPA-R1-NN_v1-r1_mon_(\d{6}-\d{6})\.nc"
tiles = (raw
    .withColumn("variable", F.regexp_extract("source", pattern, 1))
    .withColumn("scenario", F.regexp_extract("source", pattern, 2))
    .withColumn("decade", F.regexp_extract("source", pattern, 3))
    .withColumn("t_idx", F.regexp_extract(F.col("tile.metadata")["sliceDims"], r"time=(\d+)", 1).cast("int"))
    .withColumn("year", F.substring("decade", 1, 4).cast("int") + F.floor(F.col("t_idx") / 12).cast("int"))
    .withColumn("month", (F.col("t_idx") % 12 + 1).cast("int"))
    # netcdf_gbx emits the legacy 3-field tile; rebuild the v2 tile struct the rst_* functions expect
    .withColumn("tile", rx.rst_fromcontent(F.col("tile.raster"), F.lit("GTiff")))
    .withColumn("tile", rx.rst_clip("tile", F.lit(AUS_WKT), F.lit(True))))

## 2 · Normalise units with map algebra
Model output is in SI units. Analysts and ArcGIS layers expect **°C** and **mm/day**, so we convert once, in the governed layer.

In [ ]:
UNIT_EXPR = {"tasmax": ("A - 273.15", "degC"), "pr": ("A * 86400", "mm/day")}

normalised = None
for var, (expr, unit) in UNIT_EXPR.items():
    part = (tiles.where(F.col("variable") == var)
                 .withColumn("tile", rx.rst_mapalgebra(F.array("tile"), F.lit(expr)))
                 .withColumn("unit", F.lit(unit)))
    normalised = part if normalised is None else normalised.unionByName(part)

(normalised.select("variable", "unit", "scenario", "decade", "year", "month", "source", "tile")
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.silver_climate_monthly_tiles"))
spark.sql(f"COMMENT ON TABLE {FQ}.silver_climate_monthly_tiles IS "
          "'BARPA-R ACCESS-CM2 monthly rasters (0.2 deg), cropped to Australia, units normalised to degC / mm per day. One GeoBrix tile per variable x scenario x month.'")

monthly = spark.table(f"{FQ}.silver_climate_monthly_tiles")
print(f"{monthly.count():,} monthly tiles")
display(monthly.groupBy("variable", "unit", "scenario", "decade").count().orderBy("variable", "scenario", "decade"))

### Quick look: one tile's metadata and statistics (GeoBrix accessors)

In [ ]:
display(monthly.where("variable='tasmax' AND scenario='historical' AND year=2009 AND month=2")
        .select("variable", "unit", "year", "month",
                rx.rst_width("tile").alias("width_px"), rx.rst_height("tile").alias("height_px"),
                rx.rst_srid("tile").alias("srid"), rx.rst_pixelwidth("tile").alias("pixel_deg"),
                rx.rst_avg("tile").alias("mean"), rx.rst_max("tile").alias("max")))

## 3 · Reduce: decadal means (annual + summer DJF) with `rst_combineavg_agg`

In [ ]:
seasonal = (monthly.withColumn("season", F.lit("annual"))
            .unionByName(monthly.where(F.col("month").isin(12, 1, 2)).withColumn("season", F.lit("DJF"))))

decadal = (seasonal.groupBy("variable", "unit", "scenario", "decade", "season")
           .agg(rx.rst_combineavg_agg("tile").alias("tile")))   # pandas grouped-agg: keep it alone in .agg()
decadal.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.silver_climate_decadal_tiles")
decadal = spark.table(f"{FQ}.silver_climate_decadal_tiles")
display(decadal.select("variable", "scenario", "decade", "season", rx.rst_avg("tile").alias("domain_mean"))
        .orderBy("variable", "season", "scenario", "decade"))

## 4 · Change rasters vs the 2001-2010 baseline (`rst_mapalgebra` across two tiles)
Temperature change is absolute (**°C**); rainfall change is relative (**%**).

In [ ]:
base = (decadal.where(F.col("scenario") == BASELINE[0])
        .select("variable", "season", F.col("tile").alias("base_tile")))
future = decadal.where(F.col("scenario") != BASELINE[0])

change = (future.join(base, ["variable", "season"])
          .withColumn("expr", F.when(F.col("variable") == "tasmax", F.lit("A - B")).otherwise(F.lit("(A - B) / B * 100")))
          .withColumn("change_unit", F.when(F.col("variable") == "tasmax", F.lit("degC")).otherwise(F.lit("percent"))))
parts = [change.where(F.col("variable") == v)
               .withColumn("tile", rx.rst_mapalgebra(F.array("tile", "base_tile"), F.lit(e)))
         for v, e in [("tasmax", "A - B"), ("pr", "(A - B) / B * 100")]]
change = parts[0].unionByName(parts[1]).select("variable", "change_unit", "scenario", "decade", "season", "tile")
change.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_change_tiles")
change = spark.table(f"{FQ}.gold_change_tiles")
display(change.select("variable", "change_unit", "scenario", "decade", "season", rx.rst_avg("tile").alias("domain_mean_change"))
        .orderBy("variable", "season", "decade", "scenario"))

## 5 · Hand-off to ArcGIS: write Cloud-Optimised GeoTIFFs
Databricks does the heavy, repeatable compute. ArcGIS Pro / Online consumes the **COG** outputs for cartography and publishing.
COGs are written with **DEFLATE** compression for the widest GIS compatibility (GeoBrix defaults to ZSTD, which some tools can't read).
We also clip one layer to **Victoria** using the ABS state boundary.

In [ ]:
import os
out_dir = f"{VOL}/outputs/cog"
os.makedirs(out_dir, exist_ok=True)

# DEFLATE (not GeoBrix's default ZSTD) so every GIS tool can open the files, including older ArcGIS and QGIS builds
cogs = change.select("variable", "scenario", "decade", "season", rx.rst_cog_convert("tile", F.lit("DEFLATE")).alias("cog")).collect()
for r in cogs:
    name = f"{r.variable}_{r.season}_{r.scenario}_{PERIOD_LABEL[r.decade]}_change.tif"
    with open(f"{out_dir}/{name}", "wb") as fh:
        fh.write(bytes(r.cog.raster))
print(f"Wrote {len(cogs)} COGs to {out_dir}")

# Victoria clip, using the ABS state boundary (vector) as a cutline on the raster
ste = spark.read.format("shapefile_gbx").load(f"{VOL}/abs/STE_2021_AUST_SHP_GDA2020.shp.zip")
vic = ste.where("STE_NAME21 = 'Victoria'").select(F.expr("st_aswkb(st_simplify(st_geomfromwkb(geom_0), 0.01))").alias("vic_wkb"))
vic_tile = (change.where("variable='tasmax' AND season='DJF' AND scenario='ssp370' AND decade='209101-210012'")
            .crossJoin(vic).select(rx.rst_clip("tile", "vic_wkb", F.lit(True)).alias("tile")))
vic_row = vic_tile.select(rx.rst_cog_convert("tile", F.lit("DEFLATE")).alias("cog"), rx.rst_avg("tile").alias("avg")).collect()[0]
with open(f"{out_dir}/VIC_tasmax_DJF_ssp370_2091-2100_change.tif", "wb") as fh:
    fh.write(bytes(vic_row.cog.raster))
print(f"Victoria summer max-temp change, SSP3-7.0 2091-2100: mean +{vic_row.avg[0]:.2f} °C")
display(dbutils.fs.ls(out_dir.replace("/Volumes", "dbfs:/Volumes")))

## 6 · Raster → H3 grid: every monthly tile to H3 resolution 4
Once pixels are H3 cells, raster values join to **any** table (population, assets, SA2s) with a plain SQL join on `cell_id`.

In [ ]:
monthly.createOrReplaceTempView("monthly_tiles")
h3_monthly = spark.sql(f"""
    SELECT m.variable, m.unit, m.scenario, m.decade, m.year, m.month,
           g.cellID AS cell_id, g.measure AS value
    FROM monthly_tiles m,
         LATERAL gbx_rst_h3_rastertogridavg(m.tile, {H3_RES}) g
""")
h3_monthly.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.silver_h3_monthly")
spark.sql(f"COMMENT ON TABLE {FQ}.silver_h3_monthly IS 'BARPA-R monthly values aggregated (mean) to H3 resolution {H3_RES} cells via GeoBrix gbx_rst_h3_rastertogridavg'")
h3 = spark.table(f"{FQ}.silver_h3_monthly")
print(f"{h3.count():,} H3 cell-month rows, {h3.select('cell_id').distinct().count():,} distinct cells")
display(h3.where("variable='tasmax'").groupBy("scenario", "decade").agg(F.round(F.avg("value"), 2).alias("mean_tasmax_c")).orderBy("decade", "scenario"))